
# Importações

In [1]:
import os
import random
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_percentage_error as mape, root_mean_squared_error
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import MinMaxScaler

from pyESN import ESN
from shap.plots import colors
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from wsb import WSB

MODELOS = ["ESN", "MLP", "RF", "XGBoost"]
MODELOS_WSB = ["WSB-LOCAL", "WSB-GLOBAL"]
SEED = 100
SEEDS = [1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000]
HORIZONTES = [3, 6, 12]


def reset_seed(rnd_seed=SEED):
    os.environ['PYTHONHASHSEED'] = '0'
    random.seed(rnd_seed)
    np.random.seed(rnd_seed)


def calcular_rrmse(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    rmse = root_mean_squared_error(y_true, y_pred)

    mean_y_true = np.mean(y_true)

    rrmse = rmse / mean_y_true
    return rrmse


warnings.filterwarnings("ignore")
reset_seed()

C:\Users\eduar\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Carregar Datasets

In [2]:
df = pd.read_csv("./dados/dados_tratados.csv", sep=';', decimal='.')

## Normalização

In [3]:
# Repete a normalização para obtermos so scalers correspondentes
scalers = {}
dataframes = []

for campus, dados in df.groupby("CAMPUS"):
    scaler = MinMaxScaler()
    dados[["CONSUMO"]] = scaler.fit_transform(dados[["CONSUMO"]])

    scalers[campus] = scaler
    dataframes.append(dados)

df = pd.concat(dataframes, ignore_index=True)

## Criação dos Lags

In [4]:
dataframes = []

for campus, dados in df.sort_values("DATA").groupby("CAMPUS"):
    lags = {f'LAG_{i:02d}': dados['CONSUMO'].shift(i) for i in range(1, 12 + 1)}
    dados = pd.concat([dados, pd.DataFrame(lags)], axis=1)
    dados.dropna(inplace=True)
    dados["ORDEM"] = range(1, len(dados) + 1)
    dataframes.append(dados)

df = pd.concat(dataframes, ignore_index=True)
df

,CONSUMO,DATA,TEMP_MIN_MÉD_MENS,TEMP_MÉD_MIN_MENS,TEMP_MÉD_MÉD_MENS,TEMP_MÉD_MAX_MENS,TEMP_MÉD_ACC_MENS,TEMP_MAX_MÉD_MENS,PRECIPITAÇÃO_MÉD_MENS,TEMP_MIN_MIN_MENS,...,LAG_03,LAG_04,LAG_05,LAG_06,LAG_07,LAG_08,LAG_09,LAG_10,LAG_11,LAG_12
0,0.615105,2016-02-29,19,22,25,28,734,33,4,19,...,0.570938,0.553762,0.386907,0.331354,0.302547,0.384551,0.530844,0.643078,0.846886,0.502037
1,0.711047,2016-03-31,12,17,22,28,685,32,3,12,...,0.580507,0.570938,0.553762,0.386907,0.331354,0.302547,0.384551,0.530844,0.643078,0.846886
2,0.633361,2016-04-30,4,9,23,28,702,33,2,4,...,0.424204,0.580507,0.570938,0.553762,0.386907,0.331354,0.302547,0.384551,0.530844,0.643078
3,0.406291,2016-05-31,4,11,16,22,490,27,9,4,...,0.615105,0.424204,0.580507,0.570938,0.553762,0.386907,0.331354,0.302547,0.384551,0.530844
4,0.362467,2016-06-30,-1,7,14,20,408,27,3,-1,...,0.711047,0.615105,0.424204,0.580507,0.570938,0.553762,0.386907,0.331354,0.302547,0.384551
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2090,0.359224,2024-06-30,0,8,17,20,499,26,1,0,...,0.990145,0.803327,0.457985,0.822613,0.794744,0.559818,0.747059,0.839038,0.553248,0.743880
2091,1.000000,2024-07-31,1,9,13,18,413,25,6,1,...,0.680301,0.990145,0.803327,0.457985,0.822613,0.794744,0.559818,0.747059,0.839038,0.553248
2092,0.940871,2024-08-31,-3,7,15,21,468,31,1,-3,...,0.352866,0.680301,0.990145,0.803327,0.457985,0.822613,0.794744,0.559818,0.747059,0.839038
2093,0.788386,2024-09-30,10,13,20,24,591,34,3,10,...,0.359224,0.352866,0.680301,0.990145,0.803327,0.457985,0.822613,0.794744,0.559818,0.747059


## Melhores Features

In [5]:
df_features = pd.read_csv("./resultados/features/fitness_features_regressao.csv", sep=";", decimal=".")

df_features = df_features.sort_values("RRMSE").head(1).reset_index(drop=True)
df_features = pd.DataFrame(
    columns=str(df_features.iloc[0]["FEATURES"]).replace("(", '').replace(")", '').replace("'", "").split(", "))

df_features = df_features.columns

df_features

Index(['TEMP_MÉD_MIN_MENS', 'TEMP_MÉD_MÉD_MENS', 'PRECIPITAÇÃO_MÉD_MENS',
       'TEMP_MIN_MAX_MENS', 'TEMP_MAX_MIN_MENS', 'PRECIPITAÇÃO_MIN_MENS',
       'TEMP_MAX_MAX_MENS', 'DIA_DA_SEMANA_dom', 'DIA_DA_SEMANA_seg',
       'DIA_DA_SEMANA_sáb', 'DIA_DA_SEMANA_ter', 'MÊS_abr', 'MÊS_ago',
       'MÊS_fev', 'MÊS_jun', 'MÊS_mai', 'MÊS_nov', 'ANO_2021', 'ANO_2022',
       'ANO_2023', 'ANO_2015', 'ANO_2016', 'ANO_2017', 'ANO_2018', 'ANO_2019',
       'CAMPUS_ASTORGA', 'CAMPUS_CAMPO LARGO', 'CAMPUS_CAPANEMA',
       'CAMPUS_CASCAVEL', 'CAMPUS_CORONEL VIVIDA', 'CAMPUS_CURITIBA',
       'CAMPUS_GOIOERÊ', 'CAMPUS_IVAIPORÃ', 'CAMPUS_JAGUARIAÍVA',
       'CAMPUS_LONDRINA - CENTRO', 'CAMPUS_PALMAS', 'CAMPUS_PARANAGUÁ',
       'CAMPUS_PINHAIS', 'CAMPUS_TELÊMACO BORBA', 'CAMPUS_UMUARAMA',
       'CURSOS_TEC_SUBSEQUENTE', 'CURSOS_GRAD_MATUTINO',
       'CURSOS_GRAD_VESPERTINO', 'CURSOS_GRAD_NOTURNO', 'CURSOS_POS', 'FÉRIAS',
       'COVID', 'LAG_01', 'LAG_02', 'LAG_03', 'LAG_05', 'LAG_07', 'LAG_09'],


## Melhores Parâmetros

In [6]:
def get_modelo(nome, tipo_treino=None, campus=None):
    if nome == "ESN":
        return ESN(n_inputs=df_features.shape[0],
                   n_outputs=1,
                   n_reservoir=int(best["ESN"]["Reservoirs"]),
                   sparsity=best["ESN"]["Sparsity"],
                   spectral_radius=best["ESN"]["Spectral Radius"],
                   random_state=int(best["ESN"]["SEED"]))

    if nome == "MLP":
        mlp = MLPRegressor(hidden_layer_sizes=(int(best["MLP"]["Hidden Layers"]),),
                           activation=best["MLP"]["Activation"],
                           alpha=best["MLP"]["Alpha"],
                           random_state=int(best["MLP"]["SEED"]))
        return mlp

    if nome == "RF":
        return RandomForestRegressor(random_state=int(best["RF"]["SEED"]),
                                     n_estimators=int(best["RF"]["N_estimators"]),
                                     max_depth=int(best["RF"]["Max_depth"]),
                                     min_samples_split=int(best["RF"]["Min_samples_split"]),
                                     min_samples_leaf=int(best["RF"]["Min_samples_leaf"]))

    if nome == "XGBoost":
        return XGBRegressor(random_state=int(best["XGBoost"]["SEED"]),
                            n_estimators=int(best["XGBoost"]["N_estimators"]),
                            max_depth=int(best["XGBoost"]["Max_depth"]),
                            booster=best["XGBoost"]["Booster"],
                            reg_lambda=best["XGBoost"]["Lambda"],
                            reg_alpha=best["XGBoost"]["Alpha"],
                            updater="coord_descent" if best["XGBoost"]["Booster"] == "gblinear" else None)

    if nome == "WSB-LOCAL" or nome == "WSB-GLOBAL":
        strong_pred = best[f"WSB-{tipo_treino}-{campus}"]["Strong_predictor"]
        return WSB(strong_predictor=get_modelo(strong_pred),
                   weak_predictors=[get_modelo(m) for m in MODELOS if m != strong_pred],
                   weight_g=best[f"WSB-{tipo_treino}-{campus}"]["Weight_g"])

best = {}
for modelo in MODELOS:
    df_aux = pd.read_csv(
        f"./resultados/otimização - regressão/BEST-{modelo}.csv", sep=';',
        decimal='.', header=0)
    display(df_aux)
    best[modelo] = df_aux.iloc[0]

for modelo in MODELOS_WSB:
    dfs_wsb = []
    for campus in df["CAMPUS"].unique():
        df_aux = pd.read_csv(
            f"./resultados/otimização - regressão/COA-{modelo}/BEST-{modelo}-{campus}.csv", sep=';',
            decimal='.', header=0)
        dfs_wsb.append(df_aux)
        best[f"{modelo}-{campus}"] = df_aux.iloc[0]

    dfs_wsb = pd.concat(dfs_wsb, ignore_index=True)
    display(dfs_wsb)

,OTIMIZADOR,MODELO,SEED,Reservoirs,Sparsity,Spectral Radius,Fitness
0,PSO,ESN,9000,11.0,0.23,0.639,0.289


,OTIMIZADOR,MODELO,SEED,Hidden Layers,Alpha,Activation,Fitness
0,PSO,MLP,10000,220,0.979,relu,0.227


,OTIMIZADOR,MODELO,SEED,N_estimators,Max_depth,Min_samples_split,Min_samples_leaf,Fitness
0,PSO,RF,7000,15.0,232.0,11.0,4.0,0.247


,OTIMIZADOR,MODELO,SEED,N_estimators,Max_depth,Booster,Lambda,Alpha,Fitness
0,PSO,XGBoost,5000,242,119,gbtree,0.898,0.041,0.243


,OTIMIZADOR,MODELO,SEED,Strong_predictor,Weight_g,Fitness
0,COA,WSB-LOCAL-ASSIS CHATEAUBRIAND,6000,XGBoost,-1.000,0.333
1,COA,WSB-LOCAL-ASTORGA,6000,XGBoost,-1.000,0.374
2,COA,WSB-LOCAL-BARRACÃO,3000,XGBoost,-0.994,0.357
3,COA,WSB-LOCAL-CAMPO LARGO,1000,XGBoost,-0.892,0.183
4,COA,WSB-LOCAL-CAPANEMA,10000,XGBoost,-0.989,0.297
5,COA,WSB-LOCAL-CASCAVEL,10000,XGBoost,-0.989,0.306
6,COA,WSB-LOCAL-CORONEL VIVIDA,1000,XGBoost,-0.399,0.397
7,COA,WSB-LOCAL-CURITIBA,1000,XGBoost,-0.982,0.615
8,COA,WSB-LOCAL-FOZ DO IGUAÇU,10000,XGBoost,-0.989,0.375
9,COA,WSB-LOCAL-GOIOERÊ,10000,XGBoost,-0.989,0.382


,OTIMIZADOR,MODELO,SEED,Strong_predictor,Weight_g,Fitness
0,COA,WSB-GLOBAL-ASSIS CHATEAUBRIAND,10000,ESN,-0.003,0.289
1,COA,WSB-GLOBAL-ASTORGA,6000,ESN,-1.000,0.287
2,COA,WSB-GLOBAL-BARRACÃO,10000,ESN,-0.989,0.297
3,COA,WSB-GLOBAL-CAMPO LARGO,10000,ESN,-0.989,0.321
4,COA,WSB-GLOBAL-CAPANEMA,10000,ESN,-0.003,0.255
5,COA,WSB-GLOBAL-CASCAVEL,1000,ESN,-0.990,0.270
6,COA,WSB-GLOBAL-CORONEL VIVIDA,1000,ESN,-0.990,0.234
7,COA,WSB-GLOBAL-CURITIBA,1000,ESN,-0.797,0.684
8,COA,WSB-GLOBAL-FOZ DO IGUAÇU,1000,ESN,-0.990,0.316
9,COA,WSB-GLOBAL-GOIOERÊ,1000,ESN,-0.531,0.275


# Divisão dos Dados


In [7]:
dfs_treino = {}
dfs_teste = {}
for horizonte in HORIZONTES:
    treino = []
    teste = []

    for campus, dados in df.sort_values('DATA').groupby("CAMPUS"):
        dados["CAMPUS"] = campus

        dados_treino, dados_teste = train_test_split(dados, test_size=horizonte, shuffle=False)

        treino.append(dados_treino)
        teste.append(dados_teste)

    treino = pd.DataFrame(pd.concat(treino, ignore_index=True))
    teste = pd.DataFrame(pd.concat(teste, ignore_index=True))

    dfs_treino[horizonte] = treino
    dfs_teste[horizonte] = teste



# Execução dos Experimentos

In [8]:
def treino(previsor, dados_treino, features):
    x_treino = dados_treino[features].to_numpy()
    y_treino = dados_treino["CONSUMO"].to_numpy()

    previsor.fit(x_treino, y_treino)
    return previsor


def teste(previsor, historico_campus, x_teste, features, horizonte):
    historico = historico_campus[["CONSUMO"]].copy()
    x_teste = x_teste[features].copy()

    previsoes = []

    for i_test in range(horizonte):
        row = x_teste.iloc[[i_test]].copy()
        historico = pd.concat([historico, pd.DataFrame([0], columns=["CONSUMO"], index=[i_test])], axis=0)

        # Recalcula os lags conforme os valores previstos pelo modelo
        lags = pd.DataFrame({f'LAG_{i:02d}': historico["CONSUMO"].shift(i) for i in range(1, 12 + 1) if
                             f'LAG_{i:02d}' in features}).tail(1)
        row.update(lags)

        if isinstance(previsor, WSB):
            peso_t = 1
            if horizonte > 3:
                peso_t = i_test / horizonte
            prev = previsor.predict(row.to_numpy(), peso_t)[0]
        elif isinstance(previsor, ESN):
            prev = previsor.predict(row.to_numpy())[0][0]
        else:
            prev = previsor.predict(row.to_numpy())[0]

        row["CONSUMO"] = prev
        previsoes.append(prev)
        historico.update(row)

    return pd.DataFrame({"CONSUMO PREVISTO": previsoes}, index=x_teste.index)



## Treinamento Local

In [9]:
for horizonte in HORIZONTES:
    df_treino = dfs_treino[horizonte]
    for campus, dados_teste in dfs_teste[horizonte].sort_values("DATA").groupby("CAMPUS"):
        dados_teste = dados_teste.set_index('DATA')
        df_previsoes = pd.DataFrame(columns=[m for m in MODELOS + MODELOS_WSB if m != f"WSB-GLOBAL-{campus}"], index=dados_teste.index)

        for nome_modelo in df_previsoes.columns:
            # Treina o modelo com os dados do campus atual
            dados_treino = df_treino[df_treino["CAMPUS"] == campus]
            modelo = treino(get_modelo(nome_modelo, "LOCAL", campus), dados_treino, df_features)

            # Testa o modelo com os dados do campus atual
            previsoes = teste(modelo, dados_treino, dados_teste, df_features, horizonte)
            df_previsoes[[nome_modelo]] = scalers[campus].inverse_transform(previsoes[["CONSUMO PREVISTO"]])

        os.makedirs(f"resultados/regressão - local/{horizonte} meses", exist_ok=True)
        df_previsoes.to_csv(f"resultados/regressão - local/{horizonte} meses/PREVISÕES {horizonte}M - {campus}.csv",
                            sep=";", decimal=".", header=True,
                            index=True)


## Treinamento Global

In [10]:

for horizonte in HORIZONTES:
    df_treino = dfs_treino[horizonte]
    for campus, dados_teste in dfs_teste[horizonte].sort_values("DATA").groupby("CAMPUS"):
        dados_teste = dados_teste.set_index('DATA')
        df_previsoes = pd.DataFrame(columns=[m for m in MODELOS + MODELOS_WSB if m != f"WSB-LOCAL-{campus}"], index=dados_teste.index)

        for nome_modelo in df_previsoes.columns:
            # Treina o modelo com os dados de todos os campi
            modelo = treino(get_modelo(nome_modelo, "GLOBAL", campus), df_treino, df_features)

            # Testa o modelo com os dados do campus atual
            previsoes = teste(modelo, df_treino[df_treino["CAMPUS"] == campus], dados_teste, df_features, horizonte)
            df_previsoes[[nome_modelo]] = scalers[campus].inverse_transform(previsoes[["CONSUMO PREVISTO"]])

        os.makedirs(f"resultados/regressão - global/{horizonte} meses", exist_ok=True)
        df_previsoes.to_csv(f"resultados/regressão - global/{horizonte} meses/PREVISÕES {horizonte}M - {campus}.csv",
                            sep=";", decimal=".",
                            index=True)



# Análise dos Resultados

In [11]:
def ts_comparacao(campus, valor_real, valores_previstos, erros, horizonte):
    valor_real = valor_real.tail(horizonte)
    plt.figure(figsize=(12, 4.5))
    plt.rcParams['xtick.labelsize'] = 13
    plt.rcParams['ytick.labelsize'] = 14
    plt.rcParams.update({'font.size': 12})
    plt.rcParams['axes.prop_cycle'] = plt.cycler(
        color=["blue", "green", "darkgoldenrod", colors.red_rgb, "purple", "cyan", "slategrey", "coral"])

    for nome_modelo in valores_previstos.columns:
        plt.plot(valores_previstos[nome_modelo],
                 label=f"{nome_modelo} (RRMSE: {erros.loc[nome_modelo]["RRMSE"]:.2%} - MAPE: {erros.loc[nome_modelo]["MAPE"]:.2%})")

    plt.plot(valor_real["CONSUMO"], label=f"CONSUMO REAL - {campus}", color="black")

    plt.xlabel('Mês')
    plt.ylabel('Consumo (KWh)')

    ax = plt.gca()
    ax.set_facecolor('white')

    plt.grid(True, color='grey', linestyle="--", linewidth=0.5)
    plt.legend(facecolor='white')

    return plt


def medidas_desempenho(valor_real, valores_previstos, horizonte):
    df_desempenho = pd.DataFrame(columns=["MAPE", "RRMSE"], index=valores_previstos.columns)

    for nome_modelo in valores_previstos.columns:
        df_desempenho.loc[nome_modelo] = [
            mape(valor_real["CONSUMO"].tail(horizonte), valores_previstos[nome_modelo].tail(horizonte)),
            calcular_rrmse(valor_real["CONSUMO"].tail(horizonte), valores_previstos[nome_modelo].tail(horizonte)),
        ]

    return df_desempenho


## Treinamento Local

In [12]:
df_real = pd.read_csv("./dados/dados_tratados.csv", sep=';', decimal='.')

for horizonte in HORIZONTES:
    df_RRMSE = pd.DataFrame(columns=[m for m in MODELOS + MODELOS_WSB if m != "WSB-GLOBAL"], index=df_real["CAMPUS"].unique())
    df_MAPE = pd.DataFrame(columns=[m for m in MODELOS + MODELOS_WSB if m != "WSB-GLOBAL"], index=df_real["CAMPUS"].unique())

    for campus, dados in df_real.sort_values("DATA").groupby("CAMPUS"):
        try:
            consumo_previsto = pd.read_csv(
                f"resultados/regressão - local/{horizonte} meses/PREVISÕES {horizonte}M - {campus}.csv",
                sep=";", decimal=".", header=0)
        except Exception as e:
            continue

        consumo_previsto["DATA"] = pd.to_datetime(consumo_previsto["DATA"])
        consumo_previsto = consumo_previsto.set_index("DATA")

        dados["DATA"] = pd.to_datetime(dados["DATA"])
        dados = dados.set_index("DATA")

        df_desempenho = medidas_desempenho(dados, consumo_previsto, horizonte)
        df_RRMSE.loc[campus] = df_desempenho["RRMSE"]
        df_MAPE.loc[campus] = df_desempenho["MAPE"]

        plt = ts_comparacao(campus, dados, consumo_previsto, df_desempenho, horizonte)

        df_desempenho.to_csv(f"resultados/regressão - local/{horizonte} meses/RRMSE {horizonte}M {campus}.csv",
                             sep=";", decimal=".", index=True)
        plt.savefig(f"resultados/regressão - local/{horizonte} meses/PREVISÕES {horizonte}M {campus}.png",
                    bbox_inches='tight')
        plt.close()

    df_RRMSE = df_RRMSE.add_suffix(" RRMSE")
    df_MAPE = df_MAPE.add_suffix(" MAPE")

    pior_RRMSE = (df_RRMSE.eq(df_RRMSE.max(axis=1), axis=0).sum(axis=0))
    melhor_RRMSE = (df_RRMSE.eq(df_RRMSE.min(axis=1), axis=0).sum(axis=0))

    pior_MAPE = (df_MAPE.eq(df_MAPE.max(axis=1), axis=0).sum(axis=0))
    melhor_MAPE = (df_MAPE.eq(df_MAPE.min(axis=1), axis=0).sum(axis=0))

    df_medias = pd.concat([df_RRMSE, df_MAPE], axis=1)
    df_medias.loc["MÉDIAS"] = df_medias.mean()

    df_medias.loc["MELHOR"] = pd.concat([melhor_RRMSE, melhor_MAPE], axis=0)
    df_medias.loc["PIOR"] = pd.concat([pior_RRMSE, pior_MAPE], axis=0)
    df_medias.loc["SCORE (MELHOR - PIOR)"] = pd.concat([melhor_RRMSE - pior_RRMSE, melhor_MAPE - pior_MAPE], axis=0)

    df_medias.to_csv(f"resultados/regressão - local/MÉDIAS ERROS {horizonte}M.csv", sep=";", decimal=".", index=True)
    display(df_medias)



,ESN RRMSE,MLP RRMSE,RF RRMSE,XGBoost RRMSE,WSB-LOCAL RRMSE,ESN MAPE,MLP MAPE,RF MAPE,XGBoost MAPE,WSB-LOCAL MAPE
ASSIS CHATEAUBRIAND,0.311359,0.12425,0.296678,0.384678,0.265237,0.259414,0.064378,0.219504,0.289052,0.185373
ASTORGA,0.326683,0.278516,0.298948,0.334858,0.259415,0.272433,0.273608,0.27509,0.29229,0.242116
BARRACÃO,0.10952,0.159638,0.135852,0.272083,0.140259,0.096757,0.158585,0.135869,0.269534,0.13985
CAMPO LARGO,0.314383,0.286582,0.173849,0.306255,0.250791,0.275375,0.284823,0.169982,0.297039,0.236972
CAPANEMA,0.422843,0.203984,0.230291,0.166305,0.228426,0.315073,0.10328,0.171482,0.137143,0.166259
CASCAVEL,0.282968,0.178718,0.245105,0.265865,0.25033,0.253321,0.160683,0.177289,0.203063,0.180183
CORONEL VIVIDA,0.293099,0.239361,0.19786,0.218547,0.208938,0.22363,0.231081,0.16801,0.20223,0.192058
CURITIBA,0.962509,0.220101,0.167784,0.261969,0.227008,0.901734,0.133404,0.165749,0.245824,0.182182
FOZ DO IGUAÇU,0.222286,0.285499,0.118387,0.21132,0.223861,0.201441,0.247414,0.112985,0.203257,0.203255
GOIOERÊ,0.306091,0.18162,0.112599,0.281516,0.161628,0.289228,0.145535,0.099313,0.199324,0.137034


,ESN RRMSE,MLP RRMSE,RF RRMSE,XGBoost RRMSE,WSB-LOCAL RRMSE,ESN MAPE,MLP MAPE,RF MAPE,XGBoost MAPE,WSB-LOCAL MAPE
ASSIS CHATEAUBRIAND,0.32826,0.52359,0.420644,0.42222,0.372543,0.25045,0.757365,0.511374,0.479986,0.465735
ASTORGA,0.307834,0.284013,0.288267,0.300552,0.24734,0.222226,0.267246,0.242436,0.222603,0.199191
BARRACÃO,0.250062,0.209844,0.198753,0.242066,0.212325,0.208184,0.209669,0.181716,0.221636,0.194796
CAMPO LARGO,0.467923,0.425668,0.363527,0.388361,0.356853,0.601344,0.542792,0.402035,0.459732,0.44642
CAPANEMA,0.508555,0.451082,0.401643,0.338276,0.348953,0.50463,0.623115,0.503296,0.412497,0.409017
CASCAVEL,0.47233,0.419661,0.365115,0.329403,0.319788,0.591357,0.466084,0.380758,0.382373,0.368313
CORONEL VIVIDA,0.245431,0.277802,0.217247,0.205905,0.200178,0.206578,0.282892,0.167997,0.152297,0.149933
CURITIBA,0.591001,0.342616,0.312017,0.288538,0.220203,0.29812,0.344478,0.338032,0.289919,0.239084
FOZ DO IGUAÇU,0.450966,0.4306,0.432367,0.338752,0.347924,0.421524,0.416376,0.43472,0.330228,0.303093
GOIOERÊ,0.469175,0.514052,0.350443,0.53874,0.518442,0.446817,0.542512,0.346408,0.477793,0.471307


,ESN RRMSE,MLP RRMSE,RF RRMSE,XGBoost RRMSE,WSB-LOCAL RRMSE,ESN MAPE,MLP MAPE,RF MAPE,XGBoost MAPE,WSB-LOCAL MAPE
ASSIS CHATEAUBRIAND,0.280578,0.398155,0.36148,0.345205,0.347247,0.178217,0.551513,0.41961,0.41411,0.425958
ASTORGA,0.477217,0.308458,0.306776,0.334848,0.317814,0.430907,0.236664,0.228922,0.255418,0.238253
BARRACÃO,0.288687,0.229966,0.295104,0.27954,0.272587,0.233853,0.21752,0.26805,0.242562,0.240771
CAMPO LARGO,0.365603,0.334673,0.335989,0.326037,0.310423,0.358701,0.378977,0.355507,0.320603,0.336324
CAPANEMA,0.535447,0.431023,0.395956,0.340894,0.381849,0.470756,0.550737,0.436529,0.365215,0.430331
CASCAVEL,0.273705,0.298589,0.326426,0.283239,0.268733,0.283747,0.285663,0.328,0.256836,0.256419
CORONEL VIVIDA,0.345348,0.34397,0.267654,0.239236,0.244655,0.304481,0.379106,0.287886,0.250438,0.258917
CURITIBA,0.841661,0.393413,0.295968,0.354607,0.296769,0.668006,0.250875,0.264717,0.309176,0.239615
FOZ DO IGUAÇU,0.385475,0.512756,0.438348,0.485188,0.478918,0.423062,0.566678,0.510157,0.518693,0.516351
GOIOERÊ,0.25546,0.266759,0.304176,0.277809,0.265928,0.272904,0.298071,0.276366,0.281362,0.272922


## Treinamento Global

In [13]:
df_real = pd.read_csv("./dados/dados_tratados.csv", sep=';', decimal='.')

for horizonte in HORIZONTES:

    df_RRMSE = pd.DataFrame(columns=[m for m in MODELOS + MODELOS_WSB if m != "WSB-LOCAL"], index=df_real["CAMPUS"].unique())
    df_MAPE = pd.DataFrame(columns=[m for m in MODELOS + MODELOS_WSB if m != "WSB-LOCAL"], index=df_real["CAMPUS"].unique())

    for campus, dados in df_real.sort_values("DATA").groupby("CAMPUS"):
        try:
            consumo_previsto = pd.read_csv(
                f"resultados/regressão - global/{horizonte} meses/PREVISÕES {horizonte}M - {campus}.csv",
                sep=";", decimal=".", header=0)
        except Exception as e:
            continue

        consumo_previsto["DATA"] = pd.to_datetime(consumo_previsto["DATA"])
        consumo_previsto = consumo_previsto.set_index("DATA")

        dados["DATA"] = pd.to_datetime(dados["DATA"])
        dados = dados.set_index("DATA")

        df_desempenho = medidas_desempenho(dados, consumo_previsto, horizonte)
        df_RRMSE.loc[campus] = df_desempenho["RRMSE"]
        df_MAPE.loc[campus] = df_desempenho["MAPE"]

        plt = ts_comparacao(campus, dados, consumo_previsto, df_desempenho, horizonte)

        df_desempenho.to_csv(f"resultados/regressão - global/{horizonte} meses/RRMSE {horizonte}M {campus}.csv",
                             sep=";", decimal=".", index=True)
        plt.savefig(f"resultados/regressão - global/{horizonte} meses/PREVISÕES {horizonte}M {campus}.png",
                    bbox_inches='tight')
        plt.close()

    df_RRMSE = df_RRMSE.add_suffix(" RRMSE")
    df_MAPE = df_MAPE.add_suffix(" MAPE")

    pior_RRMSE = (df_RRMSE.eq(df_RRMSE.max(axis=1), axis=0).sum(axis=0))
    melhor_RRMSE = (df_RRMSE.eq(df_RRMSE.min(axis=1), axis=0).sum(axis=0))

    pior_MAPE = (df_MAPE.eq(df_MAPE.max(axis=1), axis=0).sum(axis=0))
    melhor_MAPE = (df_MAPE.eq(df_MAPE.min(axis=1), axis=0).sum(axis=0))

    df_medias = pd.concat([df_RRMSE, df_MAPE], axis=1)
    df_medias.loc["MÉDIAS"] = df_medias.mean()

    df_medias.loc["MELHOR"] = pd.concat([melhor_RRMSE, melhor_MAPE], axis=0)
    df_medias.loc["PIOR"] = pd.concat([pior_RRMSE, pior_MAPE], axis=0)
    df_medias.loc["SCORE (MELHOR - PIOR)"] = pd.concat([melhor_RRMSE - pior_RRMSE, melhor_MAPE - pior_MAPE], axis=0)

    df_medias.to_csv(f"resultados/regressão - global/MÉDIAS ERROS {horizonte}M.csv", sep=";", decimal=".", index=True)
    display(df_medias)



,ESN RRMSE,MLP RRMSE,RF RRMSE,XGBoost RRMSE,WSB-GLOBAL RRMSE,ESN MAPE,MLP MAPE,RF MAPE,XGBoost MAPE,WSB-GLOBAL MAPE
ASSIS CHATEAUBRIAND,0.335021,0.52028,0.241681,0.310252,0.334876,0.320783,0.415469,0.204475,0.210689,0.320445
ASTORGA,0.239765,0.400109,0.183562,0.11608,0.184305,0.219098,0.388658,0.176063,0.093383,0.177011
BARRACÃO,0.143302,0.255247,0.150445,0.167824,0.158565,0.125761,0.247998,0.139153,0.154716,0.14608
CAMPO LARGO,0.214925,0.346374,0.247012,0.248021,0.265138,0.184126,0.342332,0.219828,0.244664,0.258975
CAPANEMA,0.344694,0.485142,0.291509,0.278738,0.344424,0.332013,0.382279,0.261259,0.177098,0.331565
CASCAVEL,0.233249,0.370364,0.213974,0.254905,0.237447,0.169461,0.343652,0.188028,0.232148,0.216125
CORONEL VIVIDA,0.201665,0.29845,0.203325,0.222105,0.221622,0.169635,0.27545,0.178974,0.195599,0.195617
CURITIBA,0.321913,0.30471,0.057306,0.115496,0.047822,0.305627,0.297917,0.053383,0.107602,0.042045
FOZ DO IGUAÇU,0.237318,0.252213,0.111688,0.089633,0.113097,0.225089,0.235992,0.102689,0.075077,0.103781
GOIOERÊ,0.465857,0.142725,0.185727,0.263042,0.307487,0.438196,0.14076,0.193288,0.262948,0.280996


,ESN RRMSE,MLP RRMSE,RF RRMSE,XGBoost RRMSE,WSB-GLOBAL RRMSE,ESN MAPE,MLP MAPE,RF MAPE,XGBoost MAPE,WSB-GLOBAL MAPE
ASSIS CHATEAUBRIAND,0.455575,0.563616,0.531734,0.40938,0.455395,0.480224,0.772425,0.744592,0.541963,0.480161
ASTORGA,0.228303,0.192581,0.193867,0.127091,0.176777,0.188281,0.183534,0.168523,0.117226,0.146916
BARRACÃO,0.21964,0.201621,0.237855,0.179884,0.199755,0.182349,0.188269,0.194322,0.152587,0.166601
CAMPO LARGO,0.276241,0.347924,0.31567,0.359276,0.269489,0.319383,0.447986,0.40275,0.445261,0.32256
CAPANEMA,0.408605,0.516813,0.467003,0.453281,0.408501,0.489924,0.678008,0.647627,0.584114,0.489898
CASCAVEL,0.291019,0.35645,0.332701,0.378977,0.268749,0.295189,0.402957,0.405603,0.429562,0.299258
CORONEL VIVIDA,0.230479,0.214701,0.256904,0.223953,0.214552,0.179715,0.21883,0.204143,0.169478,0.169367
CURITIBA,0.287094,0.545697,0.211593,0.257673,0.251424,0.26334,0.632672,0.217248,0.281225,0.241679
FOZ DO IGUAÇU,0.256414,0.627122,0.348797,0.317584,0.235865,0.230079,0.632343,0.307016,0.259493,0.211579
GOIOERÊ,0.445886,0.654663,0.525382,0.536829,0.452468,0.356874,0.677914,0.507368,0.502124,0.381848


,ESN RRMSE,MLP RRMSE,RF RRMSE,XGBoost RRMSE,WSB-GLOBAL RRMSE,ESN MAPE,MLP MAPE,RF MAPE,XGBoost MAPE,WSB-GLOBAL MAPE
ASSIS CHATEAUBRIAND,0.383148,0.506564,0.385431,0.342546,0.383053,0.368745,0.697377,0.495534,0.437677,0.368867
ASTORGA,0.245551,0.246899,0.211425,0.194686,0.196569,0.211714,0.232863,0.18068,0.15165,0.169522
BARRACÃO,0.173661,0.138861,0.210286,0.19232,0.154232,0.129046,0.120696,0.185361,0.16917,0.128164
CAMPO LARGO,0.267975,0.404535,0.293833,0.317331,0.291592,0.296047,0.491012,0.332771,0.343311,0.333292
CAPANEMA,0.352819,0.539397,0.409939,0.430239,0.352818,0.351762,0.676248,0.461265,0.496933,0.351932
CASCAVEL,0.26059,0.332521,0.260665,0.274501,0.2603,0.245421,0.371156,0.296631,0.295241,0.280708
CORONEL VIVIDA,0.278768,0.336191,0.299007,0.269459,0.266888,0.246694,0.391226,0.259948,0.238781,0.246573
CURITIBA,0.259464,0.394579,0.299062,0.320178,0.2628,0.22737,0.412633,0.30465,0.249723,0.234407
FOZ DO IGUAÇU,0.404487,0.691073,0.43975,0.409392,0.410383,0.415203,0.805284,0.4235,0.359564,0.403531
GOIOERÊ,0.266711,0.35742,0.285289,0.270382,0.268841,0.250127,0.423674,0.30849,0.277902,0.259529
